Fetch Take Home Exam

Second: provide SQL queries

Question 1 : What are the top 5 brands by receipts scanned among users 21 and over?

Query:

WITH USERPRODUCT AS (
    SELECT C.ID,
           (STRFTIME('%Y', 'NOW') - STRFTIME('%Y', C.BIRTH_DATE)) -
           (STRFTIME('%M-%D', 'NOW') < STRFTIME('%M-%D', C.BIRTH_DATE)) AS AGE,
           B.RECEIPT_ID,
           A.BRAND
    FROM PRODUCT A
    LEFT JOIN (
        SELECT DISTINCT RECEIPT_ID, BARCODE, USER_ID 
        FROM TRANSACTIONS
        WHERE USER_ID IS NOT NULL
        GROUP BY 1, 2
    ) B ON A.BARCODE = B.BARCODE
    LEFT JOIN USER C ON B.USER_ID = C.ID
    WHERE C.ID IS NOT NULL
      AND A.BRAND IS NOT NULL 
      AND TRIM(A.BRAND) != ''
      AND A.BARCODE IS NOT NULL
      AND TRIM(A.BARCODE) != ''
),
BRANDCNT AS (
    SELECT BRAND, 
           COUNT(DISTINCT RECEIPT_ID) AS SCAN_COUNT
    FROM USERPRODUCT
    WHERE AGE >= 21 
    GROUP BY BRAND
),
BRANDRnk AS (
    SELECT BRAND,
           DENSE_RANK() OVER (ORDER BY SCAN_COUNT DESC) AS RNK
    FROM BRANDCNT
)
SELECT BRAND, RNK
FROM BRANDRnk 
WHERE RNK <= 5
ORDER BY RNK DESC;

Answer:

Since most of the user ids in transaction table do not exist in user table, it is hard to connect brand data to customer data. Based on the data availibiliity, all of the brands that have user information only scanned one time. We need a more accurate data to really understand what the the top 5 brands being scanned among users 21 and over. 
If we still want the brand , they are: ANGIE'S BOOMCHICKAPOP , 
AXE
CHEERIOS
COCA-COLA
DR TEAL'S
EQUATE
GIMME
GOOD SENSE
HERSHEY'S
LITTLE BITES
MARKETSIDE
MEIJER
PEPSI
SOUR PATCH KIDS


Question 2: Which is the leading brand in the Dips & Salsa category?

Query:

SELECT BRAND, COUNT(DISTINCT USER_ID) AS CNT
FROM PRODUCT A
LEFT JOIN (
    SELECT DISTINCT RECEIPT_ID, BARCODE, USER_ID 
    FROM TRANSACTIONS
    WHERE USER_ID IS NOT NULL
    GROUP BY 1, 2
) B ON A.BARCODE = B.BARCODE
WHERE CATEGORY_2 = 'DIPS & SALSA'
  AND BRAND IS NOT NULL
  AND TRIM(BRAND) != ''
  AND A.BARCODE IS NOT NULL
  AND TRIM(A.BARCODE) != '' 
  AND USER_ID IS NOT NULL
GROUP BY BRAND
ORDER BY CNT DESC
LIMIT 1;

Answer:

The definition of the leading brand in the Dips & Salsa category is the brand that has been purchased by the highest number of distinct customers during the months of June, July, August, and September in the year 2024.

MARKETSIDE is the leading brand in the Dips & Salsa category. It was being purchased by 16 people during the months of June, July, August, and September in the year 2024.


Quetion 3: At what percent has Fetch grown year over year?

Query:

WITH YEARLYUSERCOUNTS AS (
    SELECT 
        CAST(STRFTIME('%Y', CREATED_DATE) AS INTEGER) AS REGISTRATION_YEAR, 
        COUNT(ID) AS USER_COUNT
    FROM 
        USER
    GROUP BY 
        REGISTRATION_YEAR
    ORDER BY 
        REGISTRATION_YEAR
)
SELECT 
    A.REGISTRATION_YEAR,
    A.USER_COUNT,
    COALESCE(((A.USER_COUNT - B.USER_COUNT) * 1.0 / B.USER_COUNT) * 100, 0) AS GROWTH_PERCENTAGE
FROM 
    YEARLYUSERCOUNTS A
LEFT JOIN 
    YEARLYUSERCOUNTS B ON 
    A.REGISTRATION_YEAR = B.REGISTRATION_YEAR + 1  -- Join with the previous year
ORDER BY 
    A.REGISTRATION_YEAR;

Answer: 

Use user growth to measure Fetch grown year over year. If focus on the total registered users over time. And assume the user data that we have is the entire user register population. 

Fetch has demonstrated significant growth in its user base over the years, particularly in the earlier periods, with year-over-year increases reaching as high as 819% and continuing to show substantial growth in subsequent years. However, in the most recent year, there has been a decline of about 42%, indicating potential challenges in maintaining user engagement or market saturation. Overall, Fetch's early growth trajectory suggests a strong product-market fit, though strategies may need to be reevaluated to sustain momentum in the face of recent downturns.
